# Model Pipeline Smoke Test

Build `DBNetPP` from `config.yaml` (random weights, no checkpoint), push a
random image through it, inspect shapes at every stage, run post-processing,
and draw the resulting polygons. Nothing here depends on the dataset —
it's purely a sanity check that the wiring is correct.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
print('repo:', REPO_ROOT)

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from omegaconf import OmegaConf

from src.model import build_model
from src.postprocess import PostprocessConfig, decode_prob_map
from src.utils import draw_polygons

## 1) Load config and build model (random init)
Backbone pretrained weights are disabled here so this notebook works offline.

In [ ]:
cfg = OmegaConf.load(REPO_ROOT / 'config.yaml')
# force offline mode — no download
cfg.model.backbone.pretrained = False
print(OmegaConf.to_yaml(cfg.model))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

model = build_model(cfg).to(device)

n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'total params    : {n_total / 1e6:.2f}M')
print(f'trainable params: {n_train / 1e6:.2f}M')

## 2) Random image → check shapes stage-by-stage
Runs the backbone, neck, and head separately so any wiring bug shows its shape.

In [ ]:
IMAGE_SIZE = cfg.data.image_size   # 640
BATCH = 2

x = torch.randn(BATCH, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
print('input :', tuple(x.shape))

model.eval()
with torch.no_grad():
    feats = model.backbone(x)
    print('backbone:')
    for i, f in enumerate(feats, start=2):
        print(f'  C{i}: {tuple(f.shape)}  (stride {IMAGE_SIZE // f.shape[-1]})')

    fpn_outs = model.fpn(feats)
    print('\nfpn outputs (all at stride 4, each 64 ch):')
    for i, p in enumerate(fpn_outs, start=2):
        print(f'  P{i}: {tuple(p.shape)}')

    if model.asf is not None:
        fused = model.asf(fpn_outs)
    else:
        fused = torch.cat(fpn_outs, dim=1)
    print(f'\nneck fused: {tuple(fused.shape)}  (stride 4, 256 ch)')

    head_out = model.head(fused)
    for k, v in head_out.items():
        print(f'head {k}: {tuple(v.shape)}')

## 3) End-to-end forward in eval + train mode
Check that the output maps match the input resolution and that `train()` additionally returns the differentiable binary map.

In [ ]:
model.eval()
with torch.no_grad():
    out_eval = model(x)
print('eval keys :', list(out_eval.keys()))
for k, v in out_eval.items():
    print(f'  {k}: {tuple(v.shape)}  min={v.min().item():.3f}  max={v.max().item():.3f}')

model.train()
out_train = model(x)
print('\ntrain keys:', list(out_train.keys()))
for k, v in out_train.items():
    print(f'  {k}: {tuple(v.shape)}  min={v.min().item():.3f}  max={v.max().item():.3f}')

assert out_eval['prob'].shape[-2:] == x.shape[-2:], 'prob map must match input resolution'
assert 'binary' in out_train, 'train mode must return the differentiable binary map'
print('\n[OK] shapes and modes look right')

## 4) Backward pass works
Fake a target, compute loss, and ensure gradients flow (catches dead-code paths in the model).

In [ ]:
from src.loss import DBLoss, LossConfig

model.train()
B, H, W = BATCH, IMAGE_SIZE, IMAGE_SIZE
batch = {
    'image':       x,
    'prob_map':    torch.zeros(B, H, W, device=device),
    'prob_mask':   torch.ones(B, H, W, device=device),
    'thresh_map':  torch.zeros(B, H, W, device=device),
    'thresh_mask': torch.zeros(B, H, W, device=device),
    'score_map':   torch.ones(B, H, W, device=device),
}
# plant a fake text rectangle so prob_map has some positives
batch['prob_map'][:, 200:260, 100:500] = 1.0
batch['thresh_mask'][:, 190:270, 90:510] = 1.0
batch['thresh_map'][:, 190:270, 90:510] = 0.5

loss_fn = DBLoss(LossConfig()).to(device)

preds = model(x)
out = loss_fn(preds, batch)
out['loss'].backward()
print('loss components:')
for k in ['loss', 'l_s', 'l_t', 'l_b', 'l_bce', 'l_dice']:
    v = out[k]
    print(f'  {k:<8}: {float(v):.4f}')

# sanity: at least one grad is non-zero
some_grad_nonzero = any(p.grad is not None and p.grad.abs().sum() > 0 for p in model.parameters())
print(f'\ngradients flow: {some_grad_nonzero}')
assert some_grad_nonzero, 'no parameter received any gradient — the graph is broken'

## 5) Post-processing on a random image
With a random-init model the prob map is near-uniform, so we artificially inject
a few rectangles into the prob map before post-processing — this verifies the
`decode_prob_map` path (contour → unclip → quad) end-to-end.

In [ ]:
from scipy.ndimage import gaussian_filter

rng = np.random.default_rng(0)
fake_img = (rng.random((IMAGE_SIZE, IMAGE_SIZE, 3)) * 60 + 180).astype(np.uint8)

# hand-crafted probability map with 3 text-like blobs (pure numpy, no cv2)
prob = np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=np.float32)
for (x0, y0, x1, y1) in [(80, 120, 520, 170), (80, 220, 480, 270), (80, 320, 560, 370)]:
    prob[y0:y1, x0:x1] = 1.0
prob = gaussian_filter(prob, sigma=2.0)

post_cfg = PostprocessConfig(
    thresh=cfg.postprocess.thresh,
    box_thresh=cfg.postprocess.box_thresh,
    unclip_ratio=cfg.postprocess.unclip_ratio,
    max_candidates=cfg.postprocess.max_candidates,
    min_size=cfg.postprocess.min_size,
)
# prob map is already in original image coords, so no scale/pad
boxes, scores = decode_prob_map(prob, post_cfg, original_size=(IMAGE_SIZE, IMAGE_SIZE))
print(f'decoded {len(boxes)} boxes, scores: {[round(s, 3) for s in scores]}')

vis = draw_polygons(fake_img, boxes, scores)

fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(fake_img);          ax[0].set_title('random image');      ax[0].axis('off')
ax[1].imshow(prob, cmap='hot');  ax[1].set_title('fake prob map');     ax[1].axis('off')
ax[2].imshow(vis);               ax[2].set_title('decoded polygons');  ax[2].axis('off')
plt.tight_layout(); plt.show()

## 6) Real model output on a random image (untrained)
The polygons here are meaningless (random weights), but the path must run without error.

In [ ]:
# turn random image into a normalized tensor
from src.utils import IMAGENET_MEAN, IMAGENET_STD

img = (rng.random((IMAGE_SIZE, IMAGE_SIZE, 3)) * 255).astype(np.uint8)
img_norm = (img.astype(np.float32) / 255.0 - np.array(IMAGENET_MEAN)) / np.array(IMAGENET_STD)
tensor = torch.from_numpy(np.transpose(img_norm, (2, 0, 1))).unsqueeze(0).float().to(device)

model.eval()
with torch.no_grad():
    out = model(tensor)

prob = out['prob'][0, 0].float().cpu().numpy()
thresh = out['thresh'][0, 0].float().cpu().numpy()
print('prob   : min', prob.min(), 'max', prob.max(), 'mean', prob.mean())
print('thresh : min', thresh.min(), 'max', thresh.max(), 'mean', thresh.mean())

boxes, scores = decode_prob_map(prob, post_cfg, original_size=(IMAGE_SIZE, IMAGE_SIZE))
print(f'untrained model produced {len(boxes)} boxes (meaningless — expected)')

fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(img);                ax[0].set_title('random input');      ax[0].axis('off')
ax[1].imshow(prob, cmap='hot');   ax[1].set_title('model prob map');    ax[1].axis('off')
ax[2].imshow(thresh, cmap='hot'); ax[2].set_title('model thresh map');  ax[2].axis('off')
plt.tight_layout(); plt.show()

## 7) Throughput estimate
Rough forward-pass latency at the configured input size.

In [ ]:
N_WARMUP = 3
N_ITERS = 10
x = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)

model.eval()
with torch.no_grad():
    for _ in range(N_WARMUP):
        _ = model(x)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(N_ITERS):
        _ = model(x)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    dt = (time.perf_counter() - t0) / N_ITERS
print(f'device: {device}  batch=1  input={IMAGE_SIZE}x{IMAGE_SIZE}')
print(f'avg forward: {dt * 1000:.1f} ms  ({1 / dt:.1f} FPS)')

## 8) Checkpoint save/load round-trip
Save state_dict to a temp file and reload into a fresh model — confirms nothing in the architecture is non-serializable.

In [ ]:
import tempfile

with tempfile.NamedTemporaryFile(suffix='.pt', delete=False) as f:
    tmp_path = f.name

torch.save(model.state_dict(), tmp_path)
print(f'saved: {tmp_path}  ({Path(tmp_path).stat().st_size / 1e6:.2f} MB)')

model2 = build_model(cfg).to(device)
missing, unexpected = model2.load_state_dict(torch.load(tmp_path, map_location=device), strict=True)
print('missing   :', missing)
print('unexpected:', unexpected)

# compare a forward
model.eval(); model2.eval()
with torch.no_grad():
    a = model(x)['prob']
    b = model2(x)['prob']
diff = (a - b).abs().max().item()
print(f'max output diff after reload: {diff:.2e}')
assert diff < 1e-5, 'reload changed outputs — something is not state_dict-safe'
print('[OK] checkpoint round-trip clean')